# 📊 Couche Restitution : Analytics Avancés & Guide Metabase

---

## 🎯 Objectifs de ce Notebook
1. Valider la couche Gold à travers des requêtes SQL décisionnelles avancées (Window Functions, Pénétrations).
2. Guider pas-à-pas la connexion et la configuration de l'outil de Business Intelligence **Metabase**.
3. Réaliser des visualisations métiers clés (KPIs, Graphiques temporels, Cartes de chaleur des heures de pointe).
4. Fournir une synthèse pédagogique et des sujets d'approfondissement pour les élèves.


---
## 1. Analyses SQL Avancées sur la Couche Gold

Interrogeons notre Data Warehouse PostgreSQL avec des fonctions de fenêtrage SQL (`WINDOW FUNCTIONS`) pour observer la part de marché relative de chaque arrondissement.


In [ ]:
import os
from sqlalchemy import create_engine, text
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pg_host = os.getenv("POSTGRES_HOST", "postgres")
pg_port = os.getenv("POSTGRES_PORT", "5432")
pg_db = os.getenv("POSTGRES_DB", "gold_db")
pg_user = os.getenv("POSTGRES_USER", "postgres")
pg_pwd = os.getenv("POSTGRES_PASSWORD", "postgres123")

engine = create_engine(f"postgresql+psycopg2://{pg_user}:{pg_pwd}@{pg_host}:{pg_port}/{pg_db}")

sql_market_share = """
SELECT 
    borough,
    SUM(total_revenue) AS total_revenue,
    ROUND(SUM(total_revenue) * 100.0 / SUM(SUM(total_revenue)) OVER(), 2) AS revenue_pct,
    SUM(total_trips) AS total_trips,
    ROUND(SUM(total_trips) * 100.0 / SUM(SUM(total_trips)) OVER(), 2) AS trips_pct
FROM gold.agg_daily_taxi_metrics
GROUP BY borough
ORDER BY total_revenue DESC;
"""

with engine.connect() as conn:
    df_share = pd.read_sql(text(sql_market_share), conn)

print("📈 Part de marché des revenus et des volumes par arrondissement :")
print(df_share.to_string(index=False))


Visualisons la vitesse moyenne du trafic new-yorkais au fil de la journée :


In [ ]:
sql_speed = """
SELECT 
    pickup_hour,
    ROUND(AVG(avg_speed_mph), 2) AS avg_speed,
    SUM(total_trips) AS total_volume
FROM gold.agg_hourly_demand
GROUP BY pickup_hour
ORDER BY pickup_hour;
"""

with engine.connect() as conn:
    df_speed = pd.read_sql(text(sql_speed), conn)

plt.figure(figsize=(10, 4))
plt.plot(df_speed["pickup_hour"], df_speed["avg_speed"], marker="o", color="#2563eb", linewidth=2.5, label="Vitesse Moyenne (mph)")
plt.title("Évolution de la vitesse moyenne des taxis à New York par heure", fontsize=12, fontweight="bold")
plt.xlabel("Heure de la journée (0h - 23h)")
plt.ylabel("Vitesse (mph)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.show()


---
## 2. Guide Pas-à-Pas : Configuration de Metabase

**Metabase** est déjà démarré dans notre stack Docker Compose !

### Étape 1 : Accéder à l'interface Web
- Ouvrez votre navigateur sur : **[http://localhost:3000](http://localhost:3000)**
- Cliquez sur **"Let's get started"** (Créer un compte administrateur local, ex: `admin@formation.local` / `AdminPassword123!`).

### Étape 2 : Connecter la base de données PostgreSQL (Gold)
Renseignez les champs de connexion suivants :
- **Type de base de données** : `PostgreSQL`
- **Nom d'affichage** : `NYC Taxi Data Warehouse (Gold)`
- **Host** : `postgres` *(nom du service dans le réseau Docker)*
- **Port** : `5432`
- **Database name** : `gold_db`
- **Username** : `postgres`
- **Password** : `postgres123`
- Validez la connexion. Metabase va synchroniser automatiquement les tables du schéma `gold`.

```
        ┌──────────────────────────────────────────────┐
        │ Configuration de la Connexion Metabase       │
        ├──────────────────────────────────────────────┤
        │ Database Type : PostgreSQL                   │
        │ Name          : NYC Taxi Gold Warehouse      │
        │ Host          : postgres                     │
        │ Port          : 5432                         │
        │ Database Name : gold_db                      │
        │ Username      : postgres                     │
        │ Password      : postgres123                  │
        └──────────────────────────────────────────────┘
```


---
## 3. Idées de Visualisations Clés à Construire dans Metabase

Voici les 5 cartes à créer pour constituer le Dashboard exécutif de l'entreprise :

### 1. KPI Card : Chiffre d'Affaires Global
- **Table** : `gold.agg_daily_taxi_metrics`
- **Agrégation** : `Sum of Total Revenue`
- **Format** : Devise ($ USD).

### 2. Graphique Linéaire : Évolution Quotidienne des Revenus
- **Table** : `gold.agg_daily_taxi_metrics`
- **Axe X** : `Pickup Date`
- **Axe Y** : `Sum of Total Revenue`
- **Série groupée** : `Borough` (pour comparer Manhattan, Queens, Brooklyn).

### 3. Diagramme en Barres : Vitesse Moyenne vs Heures de Pointe
- **Table** : `gold.agg_hourly_demand`
- **Axe X** : `Pickup Hour`
- **Axe Y** : `Average of Avg Speed Mph`
- Permet de repérer instantanément le creux de vitesse entre 16h et 19h (embouteillages).

### 4. Diagramme Circulaire (Donut) : Répartition des Courses par Arrondissement
- **Table** : `gold.agg_daily_taxi_metrics`
- **Mesure** : `Sum of Total Trips`
- **Dimension** : `Borough`.

### 5. Tableau Croisé : Matrice Origine ➔ Destination
- **Table** : `gold.agg_borough_flows`
- **Lignes** : `Pickup Borough`
- **Colonnes** : `Dropoff Borough`
- **Cellules** : `Sum of Total Revenue`.


---
## 🎓 Synthèse Pédagogique & Exercices pour les Étudiants

### Ce que les élèves ont appris :
1. **Stockage Objet vs Base Relationnelle** : Pourquoi le Data Lake (MinIO) stocke l'historique brut et les tables volumineuses, tandis que PostgreSQL sert d'accélérateur pour la BI.
2. **Le Contrat de la Couche Bronze** : Ne jamais modifier les données brutes, garantir l'auditabilité et la reproductibilité.
3. **La Valeur de la Couche Silver** : Détection des données aberrantes, normalisation des formats, enrichissement vectorisé avec PySpark.
4. **La Modélisation Gold** : Concevoir des agrégats performants pour éliminer la latence côté tableau de bord.

---

### 🚀 Défis & Projets d'Approfondissement pour les Élèves :
- **Exercice 1 (Multi-mois)** : Adapter le notebook `01_ingestion_bronze` pour ingérer plusieurs mois consécutifs (`2024-01`, `2024-02`, `2024-03`) et tester la puissance du partitionnement Spark.
- **Exercice 2 (Second Dataset)** : Télécharger le jeu de données des **Green Taxis** ou **For-Hire Vehicles (FHV)** et concevoir une couche Silver unifiée.
- **Exercice 3 (Croisement Météo)** : Télécharger un jeu de données météo public (ex: NOAA sur S3) et corréler les jours de pluie new-yorkais avec la demande de taxis et les pourboires !
- **Exercice 4 (Gestion Incrémentale)** : Implémenter une logique d'ingestion incrémentale (CDC ou `MERGE`) plutôt que de recharger l'intégralité de la base à chaque exécution.
